# 02 · Phishing Guard and Coach

A two-agent assistant on an employee's Outlook inbox at Harbor Foods, a fictional food distributor. **Guard** investigates each new email with tools, records a verdict, and asks the employee before moving anything. **Coach** teaches the employee when they call it wrong.

All logic lives in the `phishing_guard` package. Each section below imports the piece it needs and shows it working. Sections follow PLAN.md: Setup, World, Classifier, Tools, Agents, Runners, Demo, Evaluation, Results, Findings.

**Status: Tuesday, Oct 6.** Sections 1 to 9 are built. Guard has all nine tools, the approval pause works both ways, both demo cases produce the expected trace twice in a row (`scripts/monday_check.py`), and `score_case` plus the run loop and the classifier baseline are in place (`scripts/tuesday_check.py`). Section 10 is written once the full run has numbers.

## 1. Setup

The OpenAI key is read from the environment via python-dotenv (`.env` is never printed). If it is missing, `ensure_openai_key()` asks with a hidden prompt. The model is `gpt-5.4-mini`, as in the class tutorial, and tracing is off.

In [1]:
import json
from phishing_guard.config import ensure_openai_key, has_openai_key, lm_model

ensure_openai_key()
print("model:", lm_model, "| key loaded:", has_openai_key())

model: gpt-5.4-mini | key loaded: True


## 2. World

Five Python structures hold all state: `USERS`, `MAILBOX`, `THREAT_INTEL`, `TRAINING_LOG`, `SECURITY_QUEUE`. `reset_world()` restores them before every run so each case starts clean. `TEST_CASES` is the answer key; no tool reads it. Mailbox and threat intel load from `data/*.json`.

In [2]:
from phishing_guard.world import (USERS, MAILBOX, THREAT_INTEL, TRAINING_LOG,
                                  SECURITY_QUEUE, TEST_CASES, reset_world)

reset_world()
print(f"{len(USERS)} users, {len(MAILBOX)} emails, "
      f"{len(THREAT_INTEL['domains'])} domain rows, {len(THREAT_INTEL['urls'])} url rows, "
      f"{len(TEST_CASES)} test cases")
for uid, u in USERS.items():
    print(f"  {uid} {u['name']:<12} {u['role']:<18} knows {u['known_domains']}")

3 users, 10 emails, 13 domain rows, 5 url rows, 10 test cases
  U-01 Dana Ruiz    Accounts Payable   knows ['apexsupply.example', 'coastalfreight.example']
  U-02 Marcus Lee   HR Coordinator     knows ['brightpathbenefits.example', 'paywell.example']
  U-03 Priya Shah   Regional Sales     knows ['metrodeli.example', 'northstar-grocers.example', 'freshmarket-co.example']


## 3. Classifier

Loads `phish_clf.joblib` trained in notebook 01. `top_words` lists the words that pushed an email's score toward phishing, which Guard sees as part of the tool result. Below, the classifier alone on the ten development emails: it should flag the obvious ones and miss the social-engineering ones that have no classic phishing vocabulary.

In [3]:
from phishing_guard.classifier import load_classifier, top_words

clf = load_classifier()
truth = {c['email_id']: c['true_label'] for c in TEST_CASES.values()}
print(f"{'email':<7}{'p(phish)':<10}{'truth':<12}top words")
for eid, e in sorted(MAILBOX.items()):
    text = e['subject'] + ' ' + e['body']
    p = float(clf.predict_proba([text])[0][1])
    print(f"{eid:<7}{p:<10.2f}{truth[eid]:<12}{top_words(text)}")

email  p(phish)  truth       top words
E-001  0.93      phishing    ['account', 'hours', 'secure', 'verify', '24 hours']
E-002  0.75      phishing    ['account', 'bank', 'payment', 'reply', 'trust']
E-003  0.22      legitimate  ['account', 'net', 'invoice', 'portal', 'apex']
E-004  0.20      phishing    ['account', 'bank', 'deposit', 'trust', 'warehouse']
E-005  0.39      phishing    ['buy', 'client', 'quick', 'major', 'favor']
E-006  0.71      phishing    ['hours', '48 hours', '99', 'pay', 'returned']
E-007  0.26      legitimate  ['account', 'best', 'rates', 'administrator', 'account manager']
E-008  0.04      legitimate  ['produce', 'spend', 'coastal', 'apex']
E-009  0.16      legitimate  ['account', 'form', 'produce', 'meet', 'ap']
E-010  0.85      phishing    ['account', 'immediately', 'kindly', 'dear', 'pending']


## 4. Tools

Guard's tools are `@function_tool` functions in `phishing_guard/tools.py`. Four gather evidence, one records the verdict, one acts on the mailbox behind a human approval, and two follow up. The ninth tool, `coach_user`, is the Coach agent wrapped with `.as_tool()` in section 5.

`check_sender_domain` and `check_url` take the value itself, so Guard has to extract the right domain or link, which is what the argument-correctness metric checks. Every tool returns data on failure and never raises.

`move_to_phishing_folder` has `needs_approval=True`, so the SDK pauses before running it, and a `@tool_input_guardrail` named `check_move` (the tutorial's `check_refund` pattern) blocks the move when the email ID is unknown, no verdict is recorded, or the verdict is legitimate. The guardrail runs after approval, right before execution, so approval cannot override it.

In [4]:
from phishing_guard.tools import FUNCTION_TOOLS

for t in FUNCTION_TOOLS:
    params = list(t.params_json_schema['properties'])
    flags = []
    if t.needs_approval:
        flags.append('needs approval')
    if t.tool_input_guardrails:
        flags.append('guardrail: check_move')
    print(f"{t.name:<24} {params!s:<40} {', '.join(flags)}")

classify_email_text      ['email_id']                             
check_sender_domain      ['domain']                               
check_url                ['url']                                  
get_user_profile         ['user_id']                              
record_verdict           ['email_id', 'verdict', 'red_flags']     
move_to_phishing_folder  ['email_id', 'reason']                   needs approval, guardrail: check_move
log_training_event       ['user_id', 'email_id', 'outcome']       
escalate_to_security     ['email_id', 'summary']                  


## 5. Agents

Coach is defined first because Guard's tool list references it, the same pattern as the tutorial's NASA manager and translator. Coach has one job: turn Guard's evidence into a lesson of 120 words or fewer for one employee about one email.

Guard's instructions follow five labeled stages (INVESTIGATE, DECIDE, ACT, FOLLOW UP, REPLY) so the trace can be checked against them. `make_guard_full()` builds Guard with all nine tools in PLAN.md order. `build_input` turns a mailbox record into the message Guard receives; it leaves out `is_simulation` and every label.

Two wording fixes were made on Monday after the first end-to-end runs, one change each, and are noted in the source: FOLLOW UP now says the move result carries the employee's decision and Guard must not reply before the stage is done; REPLY now says to copy the Coach's lesson exactly.

In [5]:
from phishing_guard.guard_agents import make_coach, make_guard_full, build_input

coach = make_coach()
guard = make_guard_full(coach)
print("Guard tools:", [t.name for t in guard.tools])
print()
print(build_input("E-002"))

Guard tools: ['classify_email_text', 'check_sender_domain', 'check_url', 'get_user_profile', 'record_verdict', 'move_to_phishing_folder', 'coach_user', 'log_training_event', 'escalate_to_security']

New email for Dana Ruiz (user ID U-01). Email ID: E-002
From: Apex Supply Billing <billing@apex-supp1y.example>
Reply-To: apex.billing@mailbox-pay.example
Subject: Updated bank details for October invoices
Links: none

Hi Dana,

Please note that Apex Supply has moved to a new bank. Starting with the October invoices, remit payment to:

Bank: Harborline Trust
Account: 4471190233
Routing: 081000210

Our previous account will be closed at month end, so please update your records before the next payment run and confirm by reply when done.

Thanks,
Marla Jensen
Billing Coordinator, Apex Supply


## 6. Runners

`show_trace` is the tutorial's helper for reading `result.new_items`. Three runners in `phishing_guard/runners.py`:

- `run_investigation(guard, email_id)`: one run, no pause handling. Sunday's check.
- `run_interactive(guard, email_id)`: the live demo. When Guard asks to move an email, a person types approve or reject, and on reject, their reason.
- `run_case(guard, case)`: evaluation. The decision and reason come from the test case, so 30 cases run without typing.

Both approval runners stop after three pauses and return every result segment. The SDK repeats pre-pause items after a resume (`scripts/probe_sdk.py`), so `collect_calls` in `phishing_guard/evaluation.py` dedupes by `call_id`.

**Sunday's end-of-day check, kept for the record.** Guard with only the five investigate/decide tools records a sensible verdict on demo cases A, B, and C.

In [6]:
from agents import Runner
from phishing_guard.trace_utils import show_trace
from phishing_guard.runners import run_investigation, run_interactive, run_case, recorded_verdict
from phishing_guard.guard_agents import make_guard, SUNDAY_ADDENDUM
from phishing_guard.tools import SUNDAY_TOOLS

guard_sunday = make_guard(SUNDAY_TOOLS, SUNDAY_ADDENDUM)
expected = {c['email_id']: c['expected_verdict'] for c in TEST_CASES.values()}
summary = []
for eid in ["E-001", "E-002", "E-003"]:
    print("=" * 72, f"\n{eid}  expected: {expected[eid]}\n" + "=" * 72)
    result = await run_investigation(guard_sunday, eid)
    show_trace(result)
    rv = recorded_verdict(eid)
    print("\nRecorded:", rv)
    summary.append((eid, expected[eid], rv['verdict']))

print("\nSUMMARY")
for eid, exp, rec in summary:
    print(f"{eid}  expected={exp:<12} recorded={rec}")

E-001  expected: phishing

Event: tool_call_item
Agent: Guard
Tool: classify_email_text
Arguments: {"email_id":"E-001"}

Event: tool_call_item
Agent: Guard
Tool: check_sender_domain
Arguments: {"domain":"m365-secure-login.example"}

Event: tool_call_item
Agent: Guard
Tool: check_url
Arguments: {"url":"https://m365-secure-login.example/reset?u=priya.shah"}

Event: tool_call_output_item
Agent: Guard
Tool result: {'phishing_probability': 0.93, 'top_words': ['account', 'hours', 'secure', 'verify', '24 hours']}

Event: tool_call_output_item
Agent: Guard
Tool result: {'reputation': 'malicious', 'age_days': 12, 'lookalike_of': None}

Event: tool_call_output_item
Agent: Guard
Tool result: {'reputation': 'malicious', 'lookalike_of': None, 'on_blocklist': True}

Event: tool_call_item
Agent: Guard
Tool: record_verdict
Arguments: {"email_id":"E-001","verdict":"phishing","red_flags":["malicious sender domain","malicious link on blocklist","urgent password-expiration request","request to verify acco

## 7. Demo

Two live cases show both branches of the approval. Each cell asks for a decision at the pause.

| Case | Email | Employee does | Proves |
| --- | --- | --- | --- |
| A | E-001, fake Microsoft 365 password-expiry notice to Priya with a blocklisted link | **approve** | Tool choice, sequence, approval gate. Email moves to Phishing, decision logged as reported. |
| B | E-002, bank-detail change to Dana from `apex-supp1y.example`, one character off her real vendor, no links, simulation flag on | **reject**, reason: That's Apex, we pay them every month. | No link check. Profile shows the real vendor domain. Coach names the swapped character and the reply-to mismatch. Logged as missed, security ticket, simulation reveal. |
| C (optional) | E-003, real invoice from `apexsupply.example` | not asked | Guard clears it with no action. |

`scripts/monday_check.py --repeat 2` runs A and B through `run_case` with scripted decisions and checks the trace against the test case and the HITL table. On Monday it passed 4/4.

In [7]:
# Demo case A: type approve at the pause.
results_a = await run_interactive(guard, "E-001")
for r in results_a:
    show_trace(r)
print("\nFolder:", MAILBOX["E-001"]["folder"], "| training log:", TRAINING_LOG, "| security queue:", SECURITY_QUEUE)


Guard wants to move E-001 to Phishing.
Reason: This email uses a malicious sender domain and a blocklisted link to pressure you into verifying your account.


Approve or reject?  approve


Verdict: phishing.

Top red flags: malicious sender domain, blocklisted link, urgent password-expiry pressure, external account-verification link.

Action taken: moved to the Phishing folder.



Event: tool_call_item
Agent: Guard
Tool: classify_email_text
Arguments: {"email_id":"E-001"}

Event: tool_call_item
Agent: Guard
Tool: check_sender_domain
Arguments: {"domain":"m365-secure-login.example"}

Event: tool_call_item
Agent: Guard
Tool: check_url
Arguments: {"url":"https://m365-secure-login.example/reset?u=priya.shah"}

Event: tool_call_item
Agent: Guard
Tool: get_user_profile
Arguments: {"user_id":"U-03"}

Event: tool_call_output_item
Agent: Guard
Tool result: {'phishing_probability': 0.93, 'top_words': ['account', 'hours', 'secure', 'verify', '24 hours']}

Event: tool_call_output_item
Agent: Guard
Tool result: {'reputation': 'malicious', 'age_days': 12, 'lookalike_of': None}

Event: tool_call_output_item
Agent: Guard
Tool result: {'reputation': 'malicious', 'lookalike_of': None, 'on

In [9]:
# Demo case B: type reject at the pause, then the reason: That's Apex, we pay them every month.
results_b = await run_interactive(guard, "E-002")
for r in results_b:
    show_trace(r)
print("\nFolder:", MAILBOX["E-002"]["folder"], "| training log:", TRAINING_LOG, "| security queue:", SECURITY_QUEUE)


Guard wants to move E-002 to Phishing.
Reason: This looks like a vendor payment-change scam using a lookalike sender domain and a reply-to mismatch.


Approve or reject?  reject
Why do you trust this email?  That's Apex, we pay them every month.


Verdict: phishing.

Top red flags: lookalike sender domain, reply-to mismatch, and a request to change bank details and confirm by reply. The email was moved to phishing, but the employee rejected the move.

1) Red flags: the lookalike sender domain “billing@apex-supp1y.example” imitating “apexsupply.example”; the “reply-to mismatch”; and the request to “update invoice payment instructions to a new bank” plus “confirm by reply.”  
2) “We pay them every month” is a common trap because familiar vendors can still be spoofed. Routine payments make urgent changes feel normal, so it’s easy to skip verification.  
3) Rule for Accounts Payable: any change to bank details must be verified through a known contact method before payment. The email is unconfirmed; call Apex using a number already on file.

This email was a training simulation sent by the security team.

Event: tool_call_item
Agent: Guard
Tool: classify_email_text
Arguments: {"email_id":"E-002"}

Event: tool_call_item
Agent: Guard
T

In [10]:
# Demo case C (optional): Guard should clear it with no pause.
results_c = await run_interactive(guard, "E-003")
print("Folder:", MAILBOX["E-003"]["folder"], "| pauses:", sum(bool(r.interruptions) for r in results_c))

Verdict: legitimate.

Top signals were clean sender domain, clean link, and no mismatch between From and Reply-To. No action was taken.
Folder: Inbox | pauses: 0


In [11]:
# The same two cases with scripted decisions, as the evaluation will run them.
from phishing_guard.evaluation import collect_calls

for case_id in ["C-01", "C-02"]:
    case = TEST_CASES[case_id]
    results = await run_case(guard, case)
    calls = collect_calls(results)
    print(f"{case_id}: {[c['name'] for c in calls]}")
    print("   folder:", MAILBOX[case['email_id']]['folder'],
          "| logged:", [r['outcome'] for r in TRAINING_LOG],
          "| escalated:", [q['email_id'] for q in SECURITY_QUEUE])

C-01: ['classify_email_text', 'check_sender_domain', 'check_url', 'get_user_profile', 'record_verdict', 'move_to_phishing_folder', 'log_training_event']
   folder: Phishing | logged: ['reported'] | escalated: []
C-02: ['classify_email_text', 'check_sender_domain', 'get_user_profile', 'record_verdict', 'move_to_phishing_folder', 'coach_user', 'log_training_event', 'escalate_to_security']
   folder: Inbox | logged: ['missed'] | escalated: ['E-002']


## 8. Evaluation

Every labeled case runs through `run_case` and is scored by `score_case` in `phishing_guard/evaluation.py`, PLAN.md's code verbatim. Eight metrics are scored by code across the three required scopes:

| Scope | Metric | Passes when |
| --- | --- | --- |
| End-to-end | `verdict_correct` | Recorded verdict equals the expected verdict |
| End-to-end | flag precision, recall, F1 | `flagged` (suspicious or phishing) against `true_label`, the proposal's metrics |
| Trajectory | `workflow_adherence` | Evidence before the verdict, verdict before the move, and the follow-up matches the HITL table for the verdict Guard recorded |
| Trajectory | `task_completion` | Final folder, training log, Coach call, and security queue match the expected verdict and scripted decision |
| Component | `tool_correctness` | Every required tool called, no forbidden tool called |
| Component | `argument_correctness` | Sender check got the From domain, each link check got a link from the email, action tools got the right email ID |
| Trajectory | `step_efficiency` | No tool called twice with the same arguments |

Adherence and completion differ on purpose: a wrong verdict followed faithfully passes the first and fails the second. The move is read from `MAILBOX`, not the trace, because a rejected call still leaves an output item. A case whose run raises becomes an error row with every metric False, so the loop finishes; on the first pass those are harness bugs to fix.

`scripts/test_score_offline.py` drives `score_case` with hand-built traces and world states (30 checks, no model). The lesson judge is scheduled for later in the week.

The cell below runs the full loop and takes several minutes (about eight model calls per case). `scripts/tuesday_check.py` does the same from the command line and saves to `results/`; set `RUN_LIVE = False` to load its latest saved run instead.

In [ ]:
from phishing_guard.evaluation import (METRIC_COLUMNS, run_all_cases, score_case,
                                       summarize, summarize_by_group,
                                       classifier_baseline, results_table,
                                       verdict_vs_classifier)
import pandas as pd
import json

RUN_LIVE = True
CASE_IDS = list(TEST_CASES)   # all cases; shorten for a smoke test, e.g. ["C-01", "C-02", "C-03"]

def progress(row):
    marks = "".join("Y" if row[m] else "." for m in METRIC_COLUMNS)
    print(f"{row['case_id']} {row['category']:<12} expected {row['expected_verdict']:<11} "
          f"got {str(row['recorded_verdict']):<11} {'ERROR' if row['error'] else marks}  ({row['tool_calls']} calls)")

if RUN_LIVE:
    rows = await run_all_cases(guard, {c: TEST_CASES[c] for c in CASE_IDS}, run_case, on_row=progress)
else:
    saved = json.loads(open("results/latest.json", encoding="utf-8").read())
    rows = saved["rows"]
    print(f"loaded {len(rows)} rows from results/latest.json ({saved['started']})")

errors = [r for r in rows if r["error"]]
print(f"\n{len(rows)} cases, {len(errors)} errors")
for r in errors:
    print(r["case_id"], r["error"].splitlines()[-1])

## 9. Results

The averages table, the classifier-only baseline on the same emails, and the per-group breakdown. The Kaggle held-out numbers come from notebook 01. The classifier sees subject + body at a 0.5 threshold, the same input the `classify_email_text` tool gives Guard, so the comparison isolates what the agent adds: sender and link reputation, the user profile, and the reasoning over them.

In [ ]:
from phishing_guard.evaluation import KAGGLE_HOLDOUT_METRICS

summary = summarize(rows)
averages = pd.DataFrame(
    [{"metric": m, "mean": summary[m]} for m in METRIC_COLUMNS]
    + [{"metric": "flag precision", "mean": summary["flag_precision"]},
       {"metric": "flag recall", "mean": summary["flag_recall"]},
       {"metric": "flag F1", "mean": summary["flag_f1"]},
       {"metric": "mean tool calls per case", "mean": summary["mean_tool_calls"]}]
).set_index("metric")
display(averages.round(2))

In [ ]:
reset_world()
baseline_rows, baseline = classifier_baseline({c: TEST_CASES[c] for c in CASE_IDS}, clf)
table = results_table(summary, baseline)
table.loc["classifier alone, Kaggle held-out"] = [KAGGLE_HOLDOUT_METRICS[k] for k in ("precision", "recall", "f1")]
display(table.round(2))
print(f"classifier ROC-AUC on these {baseline['n']} emails: {baseline['roc_auc']:.2f}  "
      f"(Kaggle held-out: {KAGGLE_HOLDOUT_METRICS['roc_auc']}, n={KAGGLE_HOLDOUT_METRICS['n']:,})")

In [ ]:
display(summarize_by_group(rows).round(2))

In [ ]:
# Per case: where the classifier and the agent disagree, and where the verdict was right but the trajectory wrong.
comparison = verdict_vs_classifier(rows, baseline_rows)
display(comparison)

df = pd.DataFrame(rows)
right_verdict_wrong_path = df[df["verdict_correct"].astype(bool) & ~df["task_completion"].astype(bool)]
print("verdict right, outcome wrong:", list(right_verdict_wrong_path["case_id"]))
failed = df[~df[METRIC_COLUMNS].astype(bool).all(axis=1)]
for _, r in failed.iterrows():
    misses = [m for m in METRIC_COLUMNS if not r[m]]
    print(f"{r['case_id']} ({r['category']}, expected {r['expected_verdict']}, got {r['recorded_verdict']}): "
          f"{', '.join(misses)}\n   {' > '.join(r['tool_sequence'])}")

## 10. Findings

*Written after the numbers exist. To fill in from the tables above:*

- Which group fails most: clear phishing, clear legitimate, or borderline.
- Agent versus classifier alone on precision, recall, and F1, and which emails make the difference.
- Cases where the verdict was right and the trajectory was wrong (words are not actions).
- Average tool calls per case, as a cost and latency number.
- Honesty notes: most emails are synthetic and written with LLM help; Guard's instructions were tuned on C-01 and C-02 (dev cases C-01 to C-10), so the score on those is optimistic.